# Chapter 2 — Encoders and Decoders

Chapter 1 built individual `EncoderLayer` and `DecoderLayer` blocks. This chapter stacks
them into complete architectures: **BERT-style encoder-only** and **GPT-2-style decoder-only**.
These two variants define the structural vocabulary used by almost every modern transformer.

## Encoder-Only Architecture (BERT)

An encoder-only model reads a complete input sequence and produces a rich contextual
representation for every token. Because no generation is happening, there is no need
for a causal mask — every token can attend to every other token in both directions.

BERT exposes **all** hidden states (one per layer) so fine-tuning tasks can pick the
most useful representation depth. Our `BertEncoder` follows this convention.

In [2]:
import torch
from transformer_book_lab.bert_encoder import BertEncoder

encoder = BertEncoder(num_layers=4, d_model=64, nhead=4, dim_feedforward=128, dropout=0.0, pre_norm=True)
encoder.eval()

x = torch.randn(2, 10, 64)  # (batch=2, seq_len=10, d_model=64)

with torch.no_grad():
    hidden_states = encoder(x)  # list of 4 tensors

print(f'Input shape:             {x.shape}')
print(f'Number of hidden states: {len(hidden_states)}  (one per layer)')
for i, hs in enumerate(hidden_states):
    print(f'  Layer {i+1} hidden state: {hs.shape}')

Input shape:             torch.Size([2, 10, 64])
Number of hidden states: 4  (one per layer)
  Layer 1 hidden state: torch.Size([2, 10, 64])
  Layer 2 hidden state: torch.Size([2, 10, 64])
  Layer 3 hidden state: torch.Size([2, 10, 64])
  Layer 4 hidden state: torch.Size([2, 10, 64])


## Decoder-Only Architecture (GPT-2)

A decoder-only model generates tokens one at a time. A **causal mask** prevents each
position from attending to future positions — position `i` can only attend to `0..i`.

Causal mask for seq_len=4 — `True` means the connection is blocked:

```
       pos0  pos1  pos2  pos3
pos0 [  F     T     T     T  ]
pos1 [  F     F     T     T  ]
pos2 [  F     F     F     T  ]
pos3 [  F     F     F     F  ]
```

Our `GPT2Decoder` accepts an optional `memory` argument for cross-attention, making it
usable as the decoder half of a full encoder-decoder model in later chapters.

In [3]:
from transformer_book_lab.gpt2_decoder import GPT2Decoder
from transformer_book_lab.masks import generate_causal_mask

# Visualise the causal mask
mask = generate_causal_mask(seq_len=6, device='cpu')
print('Causal mask (seq_len=6):')
print(mask.int())  # True=1 (blocked), False=0 (allowed)

decoder = GPT2Decoder(num_layers=6, d_model=64, nhead=4, dim_feedforward=128, dropout=0.0)
decoder.eval()

tgt = torch.randn(2, 10, 64)  # (batch=2, tgt_len=10, d_model=64)
with torch.no_grad():
    out = decoder(tgt)  # causal mask generated internally by each DecoderLayer

print(f'\nTarget input shape:   {tgt.shape}')
print(f'Decoder output shape: {out.shape}')

Causal mask (seq_len=6):
tensor([[0, 1, 1, 1, 1, 1],
        [0, 0, 1, 1, 1, 1],
        [0, 0, 0, 1, 1, 1],
        [0, 0, 0, 0, 1, 1],
        [0, 0, 0, 0, 0, 1],
        [0, 0, 0, 0, 0, 0]], dtype=torch.int32)

Target input shape:   torch.Size([2, 10, 64])
Decoder output shape: torch.Size([2, 10, 64])


## Reference Comparison — HuggingFace BERT and GPT-2

We use `transformers` to verify our implementations produce the same output *shapes*
as the canonical architectures. Values differ because HuggingFace models use pre-trained
weights; ours use random initialisation.

In [4]:
from transformers import BertModel

hf_bert = BertModel.from_pretrained('bert-base-uncased', output_hidden_states=True)
hf_bert.eval()

batch, seq_len = 2, 10
input_ids = torch.ones(batch, seq_len, dtype=torch.long)

with torch.no_grad():
    hf_out = hf_bert(input_ids)

hf_hidden = hf_out.hidden_states  # 13 tensors: embedding layer + 12 encoder layers
print(f'HuggingFace BERT: {len(hf_hidden)} hidden states, last shape: {hf_hidden[-1].shape}')

our_bert = BertEncoder(num_layers=12, d_model=768, nhead=12, dim_feedforward=3072)
our_bert.eval()
with torch.no_grad():
    our_hidden = our_bert(torch.randn(batch, seq_len, 768))

print(f'Our BertEncoder:  {len(our_hidden)} hidden states, last shape: {our_hidden[-1].shape}')
print(f'Shape match: {hf_hidden[-1].shape == our_hidden[-1].shape}')

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


HuggingFace BERT: 13 hidden states, last shape: torch.Size([2, 10, 768])
Our BertEncoder:  12 hidden states, last shape: torch.Size([2, 10, 768])
Shape match: True


In [5]:
from transformers import GPT2Model

hf_gpt2 = GPT2Model.from_pretrained('gpt2')
hf_gpt2.eval()

with torch.no_grad():
    hf_last = hf_gpt2(torch.ones(batch, seq_len, dtype=torch.long)).last_hidden_state

print(f'HuggingFace GPT-2 output shape: {hf_last.shape}')

our_gpt2 = GPT2Decoder(num_layers=12, d_model=768, nhead=12, dim_feedforward=3072)
our_gpt2.eval()
with torch.no_grad():
    our_last = our_gpt2(torch.randn(batch, seq_len, 768))

print(f'Our GPT2Decoder output shape:   {our_last.shape}')
print(f'Shape match: {hf_last.shape == our_last.shape}')

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

HuggingFace GPT-2 output shape: torch.Size([2, 10, 768])
Our GPT2Decoder output shape:   torch.Size([2, 10, 768])
Shape match: True


## Encoder-Only vs. Decoder-Only — Key Tradeoffs

| | Encoder-only (BERT) | Decoder-only (GPT-2) |
|---|---|---|
| **Attention direction** | Bidirectional — full context | Causal — past tokens only |
| **Primary task** | Understanding (classification, QA, NER) | Generation (language modelling) |
| **Training objective** | Masked Language Modelling (MLM) | Causal Language Modelling (CLM) |
| **Cross-attention** | Not present | Optional (for seq2seq decoder role) |
| **Hidden states returned** | All layers | Typically only the last |

**When bidirectional context helps**: understanding tasks where the full input is available
at inference time. BERT sees both left and right context for every token.

**When the causal constraint is necessary**: generation tasks where future tokens are
unknown. The mask enforces `P(x_t | x_1, ..., x_{t-1})`.

Modern LLMs (GPT-4, LLaMA, Claude) are decoder-only. The encoder-decoder split remains
most natural for seq2seq tasks (translation, summarisation) — covered in Ch 14–16.

## What I Learned

- **Stacking layers is simple but architecturally significant**: `BertEncoder` and
  `GPT2Decoder` are `nn.ModuleList` wrappers that define the contract later chapters plug into.
- **Returning all hidden states is BERT-specific**: fine-tuning tasks may prefer intermediate
  representations; GPT-style models only need the final layer for next-token prediction.
- **The causal mask is an upper-triangular boolean `(tgt_len, tgt_len)` tensor**: `True` at
  `[i, j]` blocks position `j` from `i`; PyTorch sends masked positions to `-inf` before softmax.
- **Decoder-only and encoder-decoder are not the same**: pure GPT-2 has no cross-attention;
  our optional `memory` argument is an architectural extension for Ch 14–16.

## Final Exercise

Wire a `BertEncoder` to a `GPT2Decoder` to form a minimal encoder-decoder model.
Pass a source sequence through the encoder, then pass a target sequence and the
encoder's final hidden state as `memory` to the decoder. Verify the output shape
is `(batch, tgt_len, d_model)`.

```python
# Hint: BertEncoder returns a list — use [-1] to get the final hidden state.
# Hint: GPT2Decoder.forward(tgt, memory=...) accepts memory as a keyword argument.
```

In [6]:
# Final Exercise: wire BertEncoder -> GPT2Decoder as a minimal encoder-decoder

enc = BertEncoder(num_layers=4, d_model=64, nhead=4, dim_feedforward=128, dropout=0.0)
dec = GPT2Decoder(num_layers=4, d_model=64, nhead=4, dim_feedforward=128, dropout=0.0)
enc.eval()
dec.eval()

src = torch.randn(2, 8, 64)  # (batch=2, src_len=8, d_model=64)
tgt = torch.randn(2, 5, 64)  # (batch=2, tgt_len=5, d_model=64)

with torch.no_grad():
    hidden_states = enc(src)       # list of 4 tensors, each (2, 8, 64)
    memory = hidden_states[-1]     # (2, 8, 64) last encoder layer
    out = dec(tgt, memory=memory)  # (2, 5, 64)

print(f"Source shape: {src.shape}")
print(f"Target shape: {tgt.shape}")
print(f"Memory shape: {memory.shape}  (last encoder layer)")
print(f"Output shape: {out.shape}")
assert out.shape == tgt.shape
print("Exercise complete: encoder-decoder pipeline verified.")


Source shape: torch.Size([2, 8, 64])
Target shape: torch.Size([2, 5, 64])
Memory shape: torch.Size([2, 8, 64])  (last encoder layer)
Output shape: torch.Size([2, 5, 64])
Exercise complete: encoder-decoder pipeline verified.
